# 00 - 环境准备与项目概览

本 notebook 帮助新人完成以下目标：
1. 理解 MiniMind-O 的项目结构与定位
2. 检查 Python / PyTorch / CUDA 环境
3. 验证关键模型文件是否已下载
4. 跑通一个最简单的模型初始化 smoke test

**阅读提示**：每个代码单元格都可以直接运行；如果出错，先看本 notebook 最后的[常见问题]部分。

## 1. 项目定位

MiniMind-O 是一个轻量级端到端 Omni 模型：
- 输入：文本、语音、图像
- 输出：文本 + 流式语音（Mimi 8 层 RVQ codes）
- 规模：约 0.1B ~ 1B dense（当前仓库主推 113M dense 与 1B dense）
- 架构：Thinker（文本主干）+ Talker（语音生成头）+ 音频/视觉投影层

核心思想：**让语音和文本在 hidden state 层面直接连通**，而不是 ASR→LLM→TTS 的级联。

## 2. 项目目录结构

先快速浏览项目根目录，了解各目录职责。

In [ ]:
import os
import sys

ROOT = "/home/genesis/Projects/minimind-o"
os.chdir(ROOT)
print("当前工作目录:", os.getcwd())

dirs = ["model", "trainer", "dataset", "scripts", "docs", "eval_results", "out", "checkpoints"]
for d in dirs:
    exists = os.path.exists(d)
    mark = "✓" if exists else "✗"
    print(f"{mark} {d:20s} exists={exists}")

## 3. 环境检查

MiniMind-O 依赖 PyTorch、transformers、funasr、pyarrow、pyaudio、librosa 等。
训练阶段最硬的依赖是：**PyTorch CUDA 可用** 和 **GPU 显存足够**。

In [ ]:
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"GPU count       : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}        : {torch.cuda.get_device_name(i)}")
        print(f"    total memory : {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

In [ ]:
# 检查关键第三方库
deps = ["transformers", "funasr", "soundfile", "librosa", "pyarrow", "PIL", "pydub"]
for dep in deps:
    try:
        mod = __import__(dep)
        ver = getattr(mod, "__version__", "unknown")
        print(f"✓ {dep:15s} {ver}")
    except ImportError as e:
        print(f"✗ {dep:15s} missing: {e}")

## 4. 预训练模型文件检查

README 中要求下载以下模型到 `model/` 目录：
- `model/SenseVoiceSmall/` —— 音频编码器
- `model/siglip2-base-p32-256-ve/` —— 视觉编码器
- `model/mimi/` —— 音频解码器（推理用）
- `model/campplus/` —— 说话人编码器
- `out/llm_768.pth` 或对应基座权重 —— 语言模型初始化

这些目录如果缺失，`MiniMindOmni` 仍可构造对象，但对应能力会退化（audio_encoder / vision_encoder 为 None）。

In [ ]:
required_paths = [
    ("model/SenseVoiceSmall", "音频编码器"),
    ("model/siglip2-base-p32-256-ve", "视觉编码器"),
    ("model/mimi", "音频解码器"),
    ("model/campplus", "说话人编码器"),
]
for p, desc in required_paths:
    exists = os.path.exists(p)
    mark = "✓" if exists else "✗"
    print(f"{mark} {p:40s} ({desc})")

# 检查基座权重
base_weights = ["out/llm_768.pth", "out/llm_2048.pth"]
for w in base_weights:
    exists = os.path.exists(w)
    print(f"{'✓' if exists else '✗'} {w} exists={exists}")

## 5. 最小模型初始化 smoke test

下面的代码只构造一个 `MiniMindOmni` 对象，不加载预训练权重、不做推理，只验证：
1. 配置对象 `OmniConfig` 能正常创建
2. 模型结构能正常实例化
3. 参数统计与 README 描述一致（113M dense）

如果这里就报错，通常是依赖库版本或路径问题。

In [ ]:
import sys
sys.path.insert(0, ROOT)

from model.model_omni import OmniConfig, MiniMindOmni

cfg = OmniConfig(
    hidden_size=768,
    num_hidden_layers=8,
    num_attention_heads=8,
    num_key_value_heads=4,
    use_moe=False,
)
print("OmniConfig 关键字段:")
print(f"  hidden_size={cfg.hidden_size}, layers={cfg.num_hidden_layers}")
print(f"  heads={cfg.num_attention_heads}, kv_heads={cfg.num_key_value_heads}, head_dim={cfg.head_dim}")
print(f"  intermediate_size={cfg.intermediate_size}")
print(f"  talker_hidden={cfg.talker_hidden_size}, talker_layers={cfg.num_talker_hidden_layers}")
print(f"  audio_vocab_size={cfg.audio_vocab_size}, audio_hidden_size={cfg.audio_hidden_size}")
print(f"  image_token_len={cfg.image_token_len}, bridge_layer={cfg.bridge_layer}")

In [ ]:
# 实例化模型（不加载权重，编码器路径缺失会打印 warning）
model = MiniMindOmni(cfg)

# 统计非冻结编码器之外的参数
def count_params(model, ignore="audio_encoder|vision_encoder"):
    import re
    total = 0
    trainable = 0
    for n, p in model.named_parameters():
        if re.search(ignore, n):
            continue
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    return total, trainable

total, trainable = count_params(model)
print(f"模型总参数（不含 audio/vision encoder）: {total / 1e6:.2f} M")
print(f"可训练参数                    : {trainable / 1e6:.2f} M")
print(f"audio_encoder is None        : {model.audio_encoder is None}")
print(f"vision_encoder is None         : {model.vision_encoder is None}")

## 6. 项目里的关键训练概念速查

| 概念 | 含义 | 对应代码 |
| --- | --- | --- |
| Thinker | 文本主干（MiniMind causal LM） | `model/model_minimind.py:MiniMindModel` |
| Talker | 语音生成模块 | `model/model_omni.py:TalkerModule` |
| bridge_layer | Thinker 哪一层的 hidden state 喂给 Talker | `OmniConfig.bridge_layer` |
| audio_proj | SenseVoice 特征 → Thinker hidden | `model/model_omni.py:MMAudioProjector` |
| vision_proj | SigLIP2 特征 → Thinker hidden | `model/model_omni.py:MMVisionProjector` |
| RVQ | Mimi 的 8 层残差向量量化 | `audio_logits` 是 8 个 logits list |
| audio_pad / stop / spk token | 音频特殊 token id：2049/2050/2051 | `OmniConfig` 默认值 |
| Muon | 用于 2D 矩阵参数的优化器 | `trainer/optimizers.py` |
| Phase-0 fixes | warmup / global loss norm / RVQ weights / val | `trainer/train_sft_omni.py` |

这些概念会在后续 notebook 中逐层展开。

## 7. 常见问题

### Q1: `torch.cuda.is_available()` 返回 False
- 检查是否装了 CPU 版 PyTorch：`python -c "import torch; print(torch.__version__)"` 应包含 `cu` 字样。
- 5090D / SM120 可能需要本地编译 PyTorch 或使用 `requirements.local-cu133-sm120.txt`。

### Q2: 模型实例化时找不到 SenseVoice / SigLIP2
- 先按 README 用 `modelscope download` 或 huggingface 下载对应目录。
- 如果只想跑结构 smoke，可以传不存在的路径，模型会打印 warning 但继续运行。

### Q3: 113M dense 参数统计不对
- 注意 `OmniConfig.intermediate_size` 默认是 `ceil(hidden*pi/64)*64`，不是 `4*hidden`。
- 113M 是 `hidden=768, layers=8, talker_hidden=768, talker_layers=4` 时的结果。

### Q4: 接下来学什么？
- 推荐顺序：`01_model_architecture.ipynb` → `02_data_flow.ipynb` → `03_training_loop.ipynb` → `04_inference_eval.ipynb`。